# Auditoria manual DENUE: registros en `revisar`

Este notebook parte de la tabla completa ya clasificada (`denue_textil_distancias.csv`), filtra solo los establecimientos que quedaron como `revisar`, y genera tres tablas de auditoria: Huejotzingo, Santa Ana Xalmimilulco y San Martin Texmelucan.

La idea es que revises cada tabla y llenes `categoria_auditada` con una de estas opciones: `alta`, `media` o `excluir`. El flujo final conserva solo `alta_relevancia_ambiental` y `media_relevancia_ambiental` para producir mapas auditados por localidad.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent

TABLES = ROOT / 'outputs' / 'tables'
MAPS = ROOT / 'outputs' / 'maps'
SRC = TABLES / 'denue_textil_distancias.csv'

REGIONS = {
    'huejotzingo': {
        'label': 'Huejotzingo',
        'source_key': 'huejotzingo',
        'audit_file': TABLES / 'auditoria_revisar_huejotzingo.csv',
    },
    'xalmimilulco': {
        'label': 'Santa Ana Xalmimilulco',
        'source_key': 'xalmimilulco',
        'audit_file': TABLES / 'auditoria_revisar_xalmimilulco.csv',
    },
    'san_martin': {
        'label': 'San Martin Texmelucan',
        'source_key': 'san_martin',
        'audit_file': TABLES / 'auditoria_revisar_san_martin.csv',
    },
}

def norm_text(value):
    return str(value).lower().replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')

def canonical_category(value):
    text = norm_text(value).strip().replace(' ', '_')
    if text in {'alta', 'alto', 'alta_relevancia', 'alta_relevancia_ambiental'}:
        return 'alta_relevancia_ambiental'
    if text in {'media', 'medio', 'mediana', 'media_relevancia', 'media_relevancia_ambiental'}:
        return 'media_relevancia_ambiental'
    if text in {'excluir', 'no', 'no_pertinente', 'descartar'}:
        return 'excluir'
    if text in {'', 'nan', 'none'}:
        return ''
    return text

display(Markdown(f'Proyecto: `{ROOT}`'))

## 1. Leer tabla completa y filtrar `revisar`

In [ ]:
denue = pd.read_csv(SRC, encoding='utf-8-sig')
revisar = denue[denue['categoria_relevancia_ambiental'].astype(str).eq('revisar')].copy()

display(Markdown(f'### Tabla completa: **{len(denue):,}** registros'))
display(Markdown(f'### Registros en revision: **{len(revisar):,}**'))
display(revisar['source_folder'].value_counts(dropna=False).rename_axis('source_folder').reset_index(name='registros'))
display(revisar.head(10))

## 2. Crear plantillas por localidad

Estas plantillas se crean solo si no existen, para no borrar tus cambios. Puedes editarlas desde Excel/LibreOffice o cargar/modificar las variables en el notebook. Llena `categoria_auditada` con `alta`, `media` o `excluir`, y usa `notas_auditoria` para justificar dudas.

In [ ]:
AUDIT_COLUMNS = [
    'id', 'clee', 'nombre_de_la_unidad_economica', 'codigo_de_la_clase_actividad_scian',
    'source_folder', 'categoria_relevancia_ambiental', 'palabras_clave_detectadas',
    'distancia_hidrografia_m', 'rango_distancia_hidrografia', 'latitud', 'longitud',
    'categoria_auditada', 'notas_auditoria'
]

templates = {}
for region_id, meta in REGIONS.items():
    mask = revisar['source_folder'].map(norm_text).str.contains(meta['source_key'], na=False)
    region_df = revisar.loc[mask].copy()
    region_df['categoria_auditada'] = ''
    region_df['notas_auditoria'] = ''
    region_df = region_df[[c for c in AUDIT_COLUMNS if c in region_df.columns]]
    path = meta['audit_file']
    if not path.exists():
        region_df.to_csv(path, index=False, encoding='utf-8-sig')
        print(f'Creada: {path.relative_to(ROOT)} ({len(region_df)} filas)')
    else:
        print(f'Ya existe, no se sobreescribe: {path.relative_to(ROOT)}')
    templates[region_id] = pd.read_csv(path, encoding='utf-8-sig')

display(Markdown('### Plantillas cargadas'))
display(pd.DataFrame([
    {'localidad': meta['label'], 'archivo': str(meta['audit_file'].relative_to(ROOT)), 'filas': len(templates[region_id])}
    for region_id, meta in REGIONS.items()
]))

## 3. Tablas para auditoria

In [ ]:
SHOW_COLUMNS = [
    'id', 'nombre_de_la_unidad_economica', 'codigo_de_la_clase_actividad_scian',
    'palabras_clave_detectadas', 'distancia_hidrografia_m', 'rango_distancia_hidrografia',
    'categoria_auditada', 'notas_auditoria'
]

for region_id, meta in REGIONS.items():
    df = templates[region_id]
    display(Markdown(f"## {meta['label']} - registros por auditar: **{len(df):,}**"))
    display(df[[c for c in SHOW_COLUMNS if c in df.columns]])

## 4. Opcional: auditar desde el notebook con diccionarios

Si prefieres editar aqui en vez de editar los CSV, escribe decisiones por `id`. Ejemplo: `3325813: ('media', 'SCIAN 313; revisar como productivo')`. Luego corre la celda para actualizar las plantillas.

In [ ]:
decisiones_huejotzingo = {
    # 3325813: ('media', 'Ejemplo: SCIAN textil, revisar como produccion/confeccion'),
}

decisiones_xalmimilulco = {
    # id: ('alta' o 'media' o 'excluir', 'nota'),
}

decisiones_san_martin = {
    # id: ('alta' o 'media' o 'excluir', 'nota'),
}

DECISIONES = {
    'huejotzingo': decisiones_huejotzingo,
    'xalmimilulco': decisiones_xalmimilulco,
    'san_martin': decisiones_san_martin,
}

for region_id, decisions in DECISIONES.items():
    if not decisions:
        continue
    path = REGIONS[region_id]['audit_file']
    df = pd.read_csv(path, encoding='utf-8-sig')
    for denue_id, value in decisions.items():
        if isinstance(value, tuple):
            category, note = value
        else:
            category, note = value, ''
        mask = df['id'].astype(str).eq(str(denue_id))
        df.loc[mask, 'categoria_auditada'] = category
        df.loc[mask, 'notas_auditoria'] = note
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'Actualizada: {path.relative_to(ROOT)}')

## 5. Consolidar auditoria

Corre esta parte despues de editar las tres plantillas. Esto pega las tres tablas en una sola salida de revision manual.

In [ ]:
audited_parts = []
for region_id, meta in REGIONS.items():
    df = pd.read_csv(meta['audit_file'], encoding='utf-8-sig')
    df['region_auditoria'] = region_id
    df['categoria_auditada_normalizada'] = df.get('categoria_auditada', '').map(canonical_category)
    audited_parts.append(df)

auditada = pd.concat(audited_parts, ignore_index=True)
out = TABLES / 'denue_revisar_categorias_auditadas.csv'
auditada.to_csv(out, index=False, encoding='utf-8-sig')

display(Markdown(f'### Tabla auditada guardada: `{out.relative_to(ROOT)}`'))
display(auditada['categoria_auditada_normalizada'].replace('', 'sin_auditar').value_counts().rename_axis('categoria').reset_index(name='registros'))
display(auditada.head(20))

## 6. Aplicar auditoria y generar mapas finales

Este paso genera `denue_categorias_auditadas.csv`, `denue_textil_auditado.gpkg` y los mapas `14_*_denue_textil_auditado_alta_media.png`, uno por localidad. No borra ni reemplaza los mapas anteriores con puntos `revisar`; crea una version posterior ya auditada, solo con `alta` y `media`.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / '07_apply_denue_audit.py')], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(ROOT / 'scripts' / '08_make_audited_maps.py')], cwd=ROOT, check=True)

display(Markdown('### Outputs principales'))
for path in [
    TABLES / 'denue_categorias_auditadas.csv',
    TABLES / 'resumen_denue_categorias_auditadas_por_localidad.csv',
    MAPS / '14_huejotzingo_denue_textil_auditado_alta_media.png',
    MAPS / '14_xalmimilulco_denue_textil_auditado_alta_media.png',
    MAPS / '14_san_martin_denue_textil_auditado_alta_media.png',
]:
    display(Markdown(f'- `{path.relative_to(ROOT)}`'))